# 04 — Gate AB-0 (i): the a-series analog solves at budget level A (R, kernel `R (y2y)`)

Mirror of the parent's Gate-0 batch (`analyses/y2y/02_solve.ipynb`) on the AB stack: binary MILP,
Gurobi, opt_gap 1e-4 + NumericFocus, **w = t on every arm** (the Gate-0-only convention — pull
1.00, only the stopping point varies). Budget = **level A** from `spec/ab_extent_v1.json`
(locked + X·unlocked = 38,055 cells, 44.7%), applied by `pr_override` — config keeps the parent
30% as the documented baseline. Spec v0.4.3 §9 AB-0; methods_log M5.

| arm | targets (w = t) | what it measures here |
|---|---|---|
| `a0_control` | none | the co-capture + lock-in **floor** of every feature under equal weights — m_soc's is the number D3's ladder reads |
| `a1_protocol` | m_soc 0.322 (frozen AB T2) | the protocol target — expected **NOT to bind** (pre-satisfied: banked 0.714 > 0.322); the parent E10 observation reproduced in the wild |
| `a4_pullcheck` | connectivity 0.80 — unreachable (cap_max @44.7% = 0.753) | pull-invariance: must be objective-equivalent to a0 |
| `a5_floor` | m_soc 0.772 (θ 2× archive lookup, D3) | does the smallest floor-clearing θ-target actually bind at the kink? |

Outputs → `runs/ab_l/gate0/<arm>/` (portfolio.tif, representation CSV, run_summary.json).
Resumable (solved arms skipped). Expect seconds–minutes per arm at 85k PU; **live internet (WLS)**.
Then `05_ab0_scenarios.ipynb`.

In [1]:
# ---- Setup: root, engine, AB manifest, budget level A ------------------------------------------
ANALYSIS <- "ab_y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))

# AB manifest: written from the AB hand-off stack (NOT the canonical aligned_stack manifest)
AB_DIR <- file.path(PROJ, "input_data", "aligned_stack_ab")
py <- file.path(PROJ, ".venv", "bin", "python")
code <- paste0("import config; print(config.write_manifest(analysis='", ANALYSIS, "', ",
               "handoff_dir=config.AB_HANDOFF_DIR, manifest_path=config.AB_HANDOFF_DIR/'manifest.json'))")
out <- suppressWarnings(system2(py, c("-c", shQuote(code)), stdout = TRUE, stderr = TRUE))
st <- attr(out, "status")
if (!is.null(st) && st != 0) stop("AB manifest refresh FAILED:\n  ", paste(out, collapse = "\n  "))
mpath <- file.path(AB_DIR, "manifest.json")
stopifnot(file.exists(mpath))
cat("AB manifest refreshed:", sub(paste0(PROJ, "/"), "", mpath), "\n")

HERE <- file.path(PROJ, "analyses", "alberta_prioritization")
EXT <- jsonlite::read_json(file.path(HERE, "spec", "ab_extent_v1.json"))
GATE0_DIR <- "analyses/alberta_prioritization/runs/ab_l/gate0"      # relative to PROJ

ctx <- pr_setup(mpath, PROJ)
# Level A budget (D-AB5) + output root -- DIRECT assignment (never modifyList; see pr_override)
ctx <- pr_override(ctx, budget_pct = EXT$budget_pct_effective,
                   results_dir = GATE0_DIR, results_subdir = "_base")

AB manifest refreshed: input_data/aligned_stack_ab/manifest.json 
prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
  override budget_pct       -> 0.4470064
  override results_dir      -> analyses/alberta_prioritization/runs/ab_l/gate0
  override results_subdir   -> _base
  EFFECTIVE targets: <none> | weight multipliers: <none>
  outputs  -> analyses/alberta_prioritization/runs/ab_l/gate0/_base


In [2]:
# ---- Ingest the AB stack (full grid, PU = non-NaN cost cells) + planning units --------------------
ctx <- modifyList(ctx, pr_ingest(ctx))
ctx <- modifyList(ctx, pr_planning_units(ctx))
stopifnot("PU count != frozen AB extent" = ctx$n_pu == EXT$n_pu,
          "locked count != frozen AB extent" = ctx$n_locked == EXT$n_locked,
          "budget cells != frozen level A" = abs(round(ctx$budget) - EXT$budget_cells) <= 1)
cat(sprintf("level A confirmed: PU %s | locked %s | budget %s cells (additions %s)\n",
            format(ctx$n_pu, big.mark = ","), format(ctx$n_locked, big.mark = ","),
            format(round(ctx$budget), big.mark = ","), format(round(ctx$budget) - ctx$n_locked, big.mark = ",")))

ingested 35 features (8 continuous + 27 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 35 features to total=100000 each (scale-invariant conditioning)
planning units: 85,133 cells | budget = 45% = 38,055 cells
locked-in [pa_mask]: 27,972 cells (32.9% of window) -- fits within budget
level A confirmed: PU 85,133 | locked 27,972 | budget 38,055 cells (additions 10,083)


In [3]:
# ---- BATCH: the four arms, resumable; w = t on every arm ---------------------------------------------
# Targets are typed here and ASSERTED against the frozen AB archive in 05 (m_soc 0.322 = T2;
# 0.772 = theta 2x lookup; 0.80 > connectivity cap_max@44.7% 0.753).
ARMS <- list(
  a0_control   = list(),
  a1_protocol  = list(irrecoverable_carbon_m_soc = 0.322),
  a4_pullcheck = list(transboundary_connectivity = 0.80),
  a5_floor     = list(irrecoverable_carbon_m_soc = 0.772)
)
OV <- list(solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
t_batch <- proc.time()[["elapsed"]]
for (arm in names(ARMS)) {
  if (file.exists(file.path(PROJ, GATE0_DIR, arm, "run_summary.json"))) {
    cat(sprintf("== %-13s already solved -- skipped\n", arm)); next
  }
  cat(sprintf("\n===================== %s =====================\n", arm))
  actx <- do.call(pr_override, c(list(ctx, targets = ARMS[[arm]],
                      feature_weight_multipliers = ARMS[[arm]],      # w = t
                      results_subdir = arm), OV))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  cat(sprintf("== %s done in %.0f s | batch elapsed %.1f min\n",
              arm, sv$timing[["elapsed"]], (proc.time()[["elapsed"]] - t_batch) / 60))
}
cat("\nAB-0 ARMS COMPLETE -- next: 05_ab0_scenarios.ipynb (kernel y2y-geo)\n")


===================== a0_control =====================
  override targets          -> {} (empty -- cleared to defaults)
  override feature_weight_multipliers -> {} (empty -- cleared to defaults)
  override results_subdir   -> a0_control
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 1
  EFFECTIVE targets: <none> | weight multipliers: <none>
  outputs  -> analyses/alberta_prioritization/runs/ab_l/gate0/a0_control
weights: 8 continuous @ 1.0 ; 27 EFG @ 0.0370 (EFG group total = 1.0)
targets: 1.00 for all 35 features (no overrides)
penalties -> connectivity=0 | boundary=0 | neighbor=0  (0 = off)


A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (all equal to 1)
││└•weights:    continuous values (between 0.03703704 and 1)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0xb3be7946
Model has 35 linear objective coefficients
Variable types: 35 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.322 and 1)
││└•weights:    continuous values (between 0.03703704 and 1)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x2fa22bb4
Model has 35 linear objective coefficients
Variable types: 35 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.8 and 1)
││└•weights:    continuous values (between 0.03703704 and 1)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0x88ba32b7
Model has 35 linear objective coefficients
Variable types: 35 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RH

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 38055)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.772 and 1)
││└•weights:    continuous values (between 0.03703704 and 1)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 36 rows, 85168 columns and 1194296 nonzeros (Min)
Model fingerprint: 0xdb87d153
Model has 35 linear objective coefficients
Variable types: 35 continuous, 85133 integer (85133 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+05]
  Objective range  [4e-02, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RH

In [4]:
# ---- quick read-back: m_soc capture per arm (the floor story at a glance) --------------------------
for (arm in names(ARMS)) {
  d <- file.path(PROJ, GATE0_DIR, arm)
  if (!file.exists(file.path(d, "run_summary.json"))) next
  rs <- jsonlite::read_json(file.path(d, "run_summary.json"))
  rep <- read.csv(file.path(d, "portfolio_representation.csv"))
  ms <- rep$relative_held[rep$feature == "irrecoverable_carbon_m_soc"]
  tg <- ARMS[[arm]][["irrecoverable_carbon_m_soc"]]
  cat(sprintf("%-13s m_soc capture %.4f%s | %5.0f s | locked %s / budget %s\n", arm, ms,
              if (is.null(tg)) " (no target)" else sprintf(" vs target %.3f", tg),
              rs$solve_seconds, format(rs$n_locked_in, big.mark = ","), format(rs$budget_cells, big.mark = ",")))
}

a0_control    m_soc capture 0.8671 (no target) |     1 s | locked 27,972 / budget 38,055
a1_protocol   m_soc capture 0.7438 vs target 0.322 |     1 s | locked 27,972 / budget 38,055
a4_pullcheck  m_soc capture 0.8671 (no target) |     1 s | locked 27,972 / budget 38,055
a5_floor      m_soc capture 0.7720 vs target 0.772 |     1 s | locked 27,972 / budget 38,055
